# Experiment 13 — weekly Beer Game decisions

**Python calculates the game; models choose orders.** Each model represents 80 fictional students in 20 four-role chains, over 24 weeks. Each week is one API request containing all 80 current decisions. Future demand is hidden.

**Run order:** full-size format pilot → 24 weekly requests → H1 probe request, for each model. This is **104 planned calls across four models**, before bounded technical retries. A failed model stops independently; successful models continue. Earlier experiment 12 results stay separate.

The **same cumulative $5 budget** covers experiments 12 and 13. It is not a fresh $5 allowance. The notebook checks the existing ledger before paid requests. It starts in free review mode.

The conditions describe imagined no pressure / strong pressure. Actual generation settings are equal between conditions. Results are descriptive: this design does not establish binding time pressure, independent student observations, structural β, or statistical significance.

[Guide](../README.md) · [Protocol](../hypotheses/PROTOCOL.md) · [Latest report](../reports/REPORT.md)

## 1. Setup

Open this **new experiment 13 notebook**, restart its kernel, then run the cells in order. If dependencies are missing, run `%pip install -r ../../../requirements.txt` in this notebook and restart the kernel. Word/docx is not required for reporting.

In [1]:
from pathlib import Path
import sys, os, json, getpass, html
import pandas as pd
from IPython.display import display, Markdown, HTML

here = Path.cwd().resolve()
candidates = [here, *here.parents, here / "Project/beer_game_experiment", here / "beer_game_experiment"]
ROOT = next((p for p in candidates if (p / "src/beergame_study/weekly_pressure.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the Beer Game project.")
sys.path.insert(0, str(ROOT / "src"))
from beergame_study import weekly_pressure as study
from beergame_study.weekly_pressure_api import Runner, estimate
from beergame_study.weekly_pressure_report import build_report

EXPERIMENT = ROOT / "experiments/13_weekly_state_api_20261021"
BUDGET_EXPERIMENT = ROOT / "experiments/12_pressure_framing_api_20261020"
print("New study:", EXPERIMENT)
print("Existing shared budget:", BUDGET_EXPERIMENT / "data/api_budget.json")

New study: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/13_weekly_state_api_20261021
Existing shared budget: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020/data/api_budget.json


## 2. Settings and cost check

Keep `RUN_API=False` to inspect everything for free. To collect, change it to `True` and keep `PHASE="all"`; no manual switch from calibration to game is needed. Use `PHASE="pilot"` only if you want to inspect the four format pilots first.

Models: `gpt-6-luna`, `gpt-5.6-luna`, `gpt-5.6-sol`, `claude-haiku-5-5`. OpenAI reasoning is `none`; Claude thinking is disabled, with low effort. These are budget/latency settings, not the pressure treatment. Output allowances are 2,048 tokens per weekly/pilot response and 4,096 per probe response.

The estimate uses synthetic states only for planning, not as experimental observations. Actual states and retries affect cost. The shared hard cap applies before every attempt; it can stop an incomplete study.

In [2]:
RUN_API = True
PHASE = "all"  # "all", "pilot", or "main" (main requires a saved accepted pilot)

run_dir = study.prepare(EXPERIMENT, BUDGET_EXPERIMENT)
plan = study.loadplan(run_dir)
print("Frozen study:", run_dir.name)
costs = pd.DataFrame(estimate(run_dir))
display(costs)
if "first_attempt_reservation_ceiling_usd" in costs:
    print(f"Planning reservations for 104 first attempts: ${costs.first_attempt_reservation_ceiling_usd.sum():.4f}")
if "stressed_input_reservation_usd" in costs:
    print(f"Large-state input sizing scenario: ${costs.stressed_input_reservation_usd.sum():.4f}")
ledger = json.loads((BUDGET_EXPERIMENT / "data/api_budget.json").read_text())
spent = sum(row["charge_usd"] for row in ledger["attempts"].values())
print(f"Already charged/reserved: ${spent:.6f}; remaining: ${ledger['cap_usd'] - spent:.6f}")
print("Main sample: 20 chains × 4 roles × 4 models = 320 fictional role identities.")
print("104 planned requests: 4 full-size format pilots + 96 weekly requests + 4 probe requests.")

Frozen study: weekly_api_c64d75aa3a03356f


,model,pilot_calls,weekly_calls,probe_calls,planned_calls,total_calls,first_pilot_reservation_usd,first_attempt_reservation_ceiling_usd,stressed_input_reservation_usd,note,existing_shared_spend_usd,cap_usd
0,gpt-6-luna,1,24,1,26,26,0.002648,0.075784,0.097166,"Full 26-call first-attempt sizing only, not re...",0.340486,5.0
1,gpt-5.6-luna,1,24,1,26,26,0.005707,0.162667,0.205430,"Full 26-call first-attempt sizing only, not re...",0.340486,5.0
2,gpt-5.6-sol,1,24,1,26,26,0.105925,3.031760,3.887030,"Full 26-call first-attempt sizing only, not re...",0.340486,5.0
3,claude-haiku-5-5,1,24,1,26,26,0.003612,0.104411,0.138622,"Full 26-call first-attempt sizing only, not re...",0.340486,5.0


Planning reservations for 104 first attempts: $3.3746
Large-state input sizing scenario: $4.3282
Already charged/reserved: $0.340486; remaining: $4.659514
Main sample: 20 chains × 4 roles × 4 models = 320 fictional role identities.
104 planned requests: 4 full-size format pilots + 96 weekly requests + 4 probe requests.


## 3. API-key input and collection

With `RUN_API=True`, this cell asks for the OpenAI and Anthropic keys in **hidden input fields**. Paste each key into its field and press Enter. Do not put keys in code cells.

Each accepted week is saved before continuing. Rerunning reuses saved accepted responses. Temporary network/rate errors get at most one budgeted retry; invalid choices stop that model and remain saved. There are no invented replacement orders or retries to obtain a desired effect. The progress bar counts accepted, failed, blocked, and remaining requests.

In [3]:
if RUN_API:
    keys = {}
    runner = None
    try:
        keys["openai"] = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ").strip()
        keys["anthropic"] = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key (hidden): ").strip()
        if not all(keys.values()):
            raise ValueError("Both keys are required; no requests sent.")
        progress_display = display(HTML("Checking saved requests…"), display_id=True)
        def show_progress(p):
            text = (f"{p['done']}/{p['total']} resolved · {p['remaining']} left · "
                    f"{p['accepted']} accepted · {p['failed']} failed · {p['skipped']} blocked · "
                    f"${p['charged_or_reserved_usd']:.4f} / $5.00")
            progress_display.update(HTML(f'<progress value="{p["done"]}" max="{p["total"]}" style="width:100%"></progress><p>{html.escape(text)}</p>'))
        runner = Runner(run_dir, keys=keys, callback=show_progress)
        execution = await runner.execute(PHASE)
        print("Saved execution status:", run_dir / "api_status.json")
    finally:
        keys.clear()
        if runner is not None:
            runner.keys.clear()
        del keys
        published = build_report(run_dir)
        print("Report updated:", published["report"])
else:
    print("Review mode: no calls and no key input. Set RUN_API=True in section 2 when ready.")

Saved execution status: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/13_weekly_state_api_20261021/data/weekly_api_c64d75aa3a03356f/api_status.json
Report updated: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/13_weekly_state_api_20261021/reports/history/b34972b65bf5/REPORT.md


## 4. Results and report

Run this even after an interruption. Conclusions appear first. Missing weeks stay missing; only full 24-week games enter H2. A valid null or opposite result is kept. The workbook and CSV files include every accepted weekly order, Python state, cost, probe choice, hypothesis comparison and recorded API attempt.

In [4]:
published = build_report(run_dir)
summary = published["summary"]
display(pd.DataFrame(summary["model_status"]))
display(pd.DataFrame(summary["hypotheses"]))
report_link = os.path.relpath(published["report"], EXPERIMENT / "notebooks")
workbook_link = os.path.relpath(published["workbook"], EXPERIMENT / "notebooks")
display(Markdown(f"[Full report]({report_link}) · [Excel results]({workbook_link})"))
display(Markdown((EXPERIMENT / "reports/REPORT.md").read_text().replace("](history/", "](../reports/history/")))
print("All raw responses and exact requests are linked from the report.")

,model,accepted_pilot_responses,expected_pilot_responses,accepted_weeks,expected_weeks,accepted_probe_responses,expected_probe_responses,complete,accepted_main_responses,inference_available
0,gpt-6-luna,1,1,24,24,1,1,True,24,False
1,gpt-5.6-luna,1,1,24,24,1,1,True,24,False
2,gpt-5.6-sol,1,1,24,24,1,1,True,24,False
3,claude-haiku-5-5,1,1,24,24,1,1,True,24,False


,model,hypothesis,metric,expected_sign,paired_blocks,mean_strong_minus_no,descriptive_direction,p_raw,p_holm_8,paired_bootstrap_95,inference_available,literal_binding_time_test
0,gpt-6-luna,H1,sensitivity,-1,10,0.000000,zero,None,None,None,False,False
1,gpt-6-luna,H2,total_cost,1,10,0.000000,zero,None,None,None,False,False
2,gpt-5.6-luna,H1,sensitivity,-1,10,-0.008333,predicted,None,None,None,False,False
3,gpt-5.6-luna,H2,total_cost,1,10,17.850000,predicted,None,None,None,False,False
4,gpt-5.6-sol,H1,sensitivity,-1,10,0.000000,zero,None,None,None,False,False
5,gpt-5.6-sol,H2,total_cost,1,10,0.000000,zero,None,None,None,False,False
6,claude-haiku-5-5,H1,sensitivity,-1,10,-0.014583,predicted,None,None,None,False,False
7,claude-haiku-5-5,H2,total_cost,1,10,-19.400000,opposite,None,None,None,False,False


[Full report](../reports/history/b34972b65bf5/REPORT.md) · [Excel results](../analysis/weekly_api_c64d75aa3a03356f/b34972b65bf5/RESULTS.xlsx)

# Experiment 13: latest findings

1. **Completion:** 4/4 models complete; 96/96 weekly batches, 4/4 pilots and 4/4 probe batches accepted.
2. **H1/H2:** gpt-6-luna H1: +0.0000; gpt-6-luna H2: +0.0000; gpt-5.6-luna H1: -0.0083; gpt-5.6-luna H2: +17.8500; gpt-5.6-sol H1: +0.0000; gpt-5.6-sol H2: +0.0000; claude-haiku-5-5 H1: -0.0146; claude-haiku-5-5 H2: -19.4000. These are descriptive strong-minus-no-pressure differences.
3. **Accounting:** Python calculates all inventory, deliveries, backlog and costs from accepted orders. Missing decisions remain missing.
4. **Meaning:** Imagined pressure framing, not an enforced deadline. H1 is a matched-state sensitivity proxy, not structural β. Batched fictional chains are dependent; no p-values are reported.
5. **Shared budget:** $1.179072 charged/reserved across experiments 12 and 13; $3.820928 remains within the original $5 cap.

[Full report and detailed data](../reports/history/b34972b65bf5/REPORT.md).


All raw responses and exact requests are linked from the report.


## 5. See the prompts

This preview shows the first main week before any choices. Later weekly prompts depend on previously accepted orders. Exact sent messages and schemas are saved as each request is prepared. The full demand schedule remains in the local frozen plan and is never included in a weekly prompt.

In [5]:
preview = study.build_task(plan, study.MODELS[0], "main", week=1, accepted={})
print(study.prompt_text(preview))
print("Exact request records:", run_dir)
print("Prompt archive:", EXPERIMENT / "prompts" / run_dir.name)

Choose Beer Game orders for the supplied fictional undergraduate business students.
There are four students per chain: r=retailer,w=wholesaler,d=distributor,f=factory.
They have introductory business knowledge and no specialist inventory training.
Assigned personalities describe ordinary coursework, not an ordering strategy.
Each student tries to minimize its team's inventory-holding and backlog costs.
Apply the classroom situation assigned to that chain. These imagined situations
do not enforce or measure actual model time or student psychological pressure.
Do not invent demographic traits, deliberate errors, a desired group pattern or
artificial variation. Equal decisions, zero orders and unchanged orders are allowed.

Rules: holding costs $0.50 per remaining case/week; backlog costs $1 per unfilled
case/week. There is no revenue, manufacturing charge, ordering fee, bulk discount
or capacity limit. Each order is a nonnegative integer number of cases. Retailer,
wholesaler and distribu